In [1]:
import pandas as pd
df_loan = pd.read_csv('loan_approval_dataset.csv')
df_loan.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [2]:
df_loan2=df_loan.copy()


In [3]:
df_loan2.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [6]:
df_loan2["loan_status"] = df_loan2["loan_status"].str.strip()
df_loan2["loan_status_encoded"] = df_loan2["loan_status"].map({"Rejected":0, "Approved":1})
df_loan2 = df_loan2.drop(columns=["loan_status"])
df_loan2["self_employed"] = df_loan2["self_employed"].str.strip()
df_loan2["self_employed_encoded"] = df_loan2["self_employed"].map({"No":0, "Yes":1})
df_loan2 = df_loan2.drop(columns=["self_employed"])
df_loan2["education"] = df_loan2["education"].str.strip()
df_loan2["education_encoded"] = df_loan2["education"].map({"Graduate":0, "Not Graduate":1})
df_loan2 = df_loan2.drop(columns=["education"])

In [7]:
df_loan2 = df_loan2.drop(columns=["loan_id"])

In [5]:
df_loan2.columns = df_loan2.columns.str.strip()
df_loan2["loan_to_income"] = df_loan2["loan_amount"] / (df_loan2["income_annum"]+1e-6)
df_loan2["total_assets"] = (df_loan2["residential_assets_value"] +
                            df_loan2["commercial_assets_value"] +
                            df_loan2["luxury_assets_value"] +
                            df_loan2["bank_asset_value"])
df_loan2["assets_to_loan"] = df_loan2["total_assets"] / (df_loan2["loan_amount"]+1e-6)

In [8]:
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

X = df_loan2.drop(columns=["loan_status_encoded"])
y = df_loan2["loan_status_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

with mlflow.start_run():

    model = DecisionTreeClassifier(
        max_depth=7,
        min_samples_split=5,
        min_samples_leaf=10,
        random_state=42,
        class_weight="balanced"
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1-score:", f1)

    # تسجيل ال parameters
    mlflow.log_param("max_depth", 7)
    mlflow.log_param("min_samples_split", 5)
    mlflow.log_param("min_samples_leaf", 10)

    # تسجيل ال metrics
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)

    # حفظ الموديل
    mlflow.sklearn.log_model(model, "decision_tree_model")

2026/03/12 02:06:17 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/03/12 02:06:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instea

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0


2026/03/12 02:06:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
import mlflow
# السطر ده هو الأهم: بيخلي الـ Notebook يكتب في قاعدة البيانات اللي في الصورة
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# ده بيخليك تلاقي نتائجك تحت اسم واضح بدل "Default"
mlflow.set_experiment("Loan_Prediction_Exp")

<Experiment: artifact_location='file:c:/Users/Mahmoud/Desktop/VS CODE/loan_approval/mlruns/1', creation_time=1773271853410, experiment_id='1', last_update_time=1773271853410, lifecycle_stage='active', name='Loan_Prediction_Exp', tags={}, workspace='default'>

In [ ]:
%pip install mlflow


  Using cached mlflow-3.10.1-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_skinny-3.10.1-py3-none-any.whl.metadata (32 kB)
  Using cached mlflow_tracing-3.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached databricks_sdk-0.98.0-py3-none-any.whl.metadata (40 kB)
  Using cached fastapi-0.135.1-py3-none-any.whl.metadata (30 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_proto-1.40.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached sqlparse-0.5.5-py3-none-any.w

In [ ]:
import mlflow
import os

# 1. تحديد مسار قاعدة البيانات في الفولدر الحالي
db_path = os.path.abspath("mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

# 2. بدء تسجيل تجربة جديدة
mlflow.set_experiment("Final_Test")

with mlflow.start_run():
    mlflow.log_metric("accuracy", 0.95)
    mlflow.log_param("model_type", "DecisionTree")
    print("Done! Data sent to:", db_path)

2026/03/12 01:35:56 INFO mlflow.tracking.fluent: Experiment with name 'Final_Test' does not exist. Creating a new experiment.


Done! Data sent to: c:\Users\Mahmoud\Desktop\VS CODE\loan_approval\mlflow.db
